In [7]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import skew, kurtosis
from astropy.timeseries import BoxLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [8]:
df = pd.read_csv("paired_simulation_labels_combined.csv")

In [9]:
# ==========================================
# 1. I/O and Preprocessing Utilities
# ==========================================

def load_classifier_lightcurve(folder_path):
    """Load a light curve, prioritizing the newly drift-corrected data."""
    # Look for corrected data first
    dat_files = glob.glob(os.path.join(folder_path, "*_driftcorrected.dat"))
    if not dat_files:
        # Fallback to original data if corrected doesn't exist
        dat_files = glob.glob(os.path.join(folder_path, "*.dat"))
        
    if dat_files:
        try:
            data = np.genfromtxt(dat_files[0])
            if data.ndim == 2 and data.shape[1] >= 2:
                return data[:, 0], data[:, 1]
        except Exception:
            pass
    return None, None

def lightcurve_to_ppm(time, flux, max_points=6_000):
    """Clean, median-center, and downsample the light curve."""
    time = np.asarray(time, dtype=float)
    flux = np.asarray(flux, dtype=float)
    
    # Filter for finite values
    finite = np.isfinite(time) & np.isfinite(flux)
    t = time[finite]
    f = flux[finite]

    # Ensure chronological order
    order = np.argsort(t)
    t = t[order]
    f = f[order]

    # Center around 0
    y_ppm = f - np.nanmedian(f)

    # Downsample for performance if necessary
    if max_points is not None and len(t) > max_points:
        idx = np.linspace(0, len(t) - 1, max_points).astype(int)
        t = t[idx]
        y_ppm = y_ppm[idx]

    # Remove duplicate timestamps
    unique = np.concatenate([[True], np.diff(t) > 0])
    return t[unique], y_ppm[unique]

def robust_noise_ppm(y_ppm):
    """Calculate robust standard deviation using Median Absolute Deviation."""
    dy = np.diff(y_ppm)
    if len(dy) == 0:
        return np.nan
    sigma = 1.4826 * np.nanmedian(np.abs(dy - np.nanmedian(dy))) / np.sqrt(2.0)
    return float(sigma) if np.isfinite(sigma) and sigma > 0 else float(np.nanstd(y_ppm))




In [ ]:
# ==========================================
# 2. Unified Feature Extraction
# ==========================================

def extract_all_features(time, flux, max_points=6_000):
    """
    Extracts all statistical, morphological, and periodic features 
    in a single pass to prevent redundant array operations.
    """
    t_days, y_ppm = lightcurve_to_ppm(time, flux, max_points=max_points)
    
    if len(y_ppm) < 20:
        raise ValueError("Insufficient data points after cleaning.")

    sigma = max(robust_noise_ppm(y_ppm), 1.0)
    cadence = float(np.nanmedian(np.diff(t_days))) if len(t_days) > 1 else np.nan
    
    # Pre-compute arrays for speed
    positive = np.clip(y_ppm, 0.0, None)
    above_3 = y_ppm > 3.0 * sigma
    above_5 = y_ppm > 5.0 * sigma
    percentiles = np.nanpercentile(y_ppm, [1, 5, 10, 50, 90, 95, 99])
    
    # Vectorized Run-Length Calculation (Replacing the slow for-loop)
    padded = np.pad(above_3, (1, 1), mode='constant', constant_values=False)
    diffs = np.diff(padded.astype(int))
    starts = np.where(diffs == 1)[0]
    ends = np.where(diffs == -1)[0]
    runs = ends - starts
    longest_run = int(np.max(runs)) if len(runs) > 0 else 0

    # A. Base Statistical Features
    features = {
        "flux_std_ppm": float(np.nanstd(y_ppm)),
        "skew": float(skew(y_ppm, nan_policy="omit")),
        "kurtosis": float(kurtosis(y_ppm, nan_policy="omit")),
        "negative_outlier_fraction": float(np.mean(y_ppm < -3.0 * sigma)),
        "p01_ppm": float(percentiles[0]),
        "p05_ppm": float(percentiles[1]),
        "p10_ppm": float(percentiles[2]),
        "p90_ppm": float(percentiles[4]),
        "p95_ppm": float(percentiles[5]),
        "p99_ppm": float(percentiles[6]),
    }

    # B. Flare Morphological Features
    max_idx = int(np.nanargmax(y_ppm))
    local_window = y_ppm[max(0, max_idx - 20) : min(len(y_ppm), max_idx + 21)]
    
    features.update({
        "flare_max_snr": float(np.nanmax(y_ppm) / sigma),
        "flare_positive_area": float(np.nansum(positive) * cadence) if np.isfinite(cadence) else np.nan,
        "flare_positive_frac_3sig": float(np.mean(above_3)),
        "flare_positive_frac_5sig": float(np.mean(above_5)),
        "flare_longest_run_pts": float(longest_run),
        "flare_longest_run_days": float(longest_run * cadence) if np.isfinite(cadence) else np.nan,
        "flare_peak_contrast": float(np.nanmax(local_window) - np.nanmedian(local_window)) if len(local_window) else np.nan,
    })

    # C. BLS Periodic Features
    span = float(np.nanmax(t_days) - np.nanmin(t_days))
    max_period = min(100.0, 0.8 * span)
    min_period = min(2.0, max_period / 2.0)
    
    if np.isfinite(max_period) and max_period > min_period:
        try:
            model = BoxLeastSquares(t_days, y_ppm, dy=np.full_like(y_ppm, sigma))
            result = model.power(np.geomspace(min_period, max_period, 160), (0.08, 0.16, 0.32, 0.64), objective="snr")
            best = int(np.nanargmax(result.power))
            
            depth = float(np.ravel(result.depth)[best])
            depth_err = float(np.ravel(result.depth_err)[best])
            
            features.update({
                "bls_power": float(np.ravel(result.power)[best]),
                "bls_period": float(np.ravel(result.period)[best]),
                "bls_duration": float(np.ravel(result.duration)[best]),
                "bls_depth_ppm": depth,
                "bls_depth_snr": float(abs(depth) / depth_err) if depth_err > 0 else np.nan,
            })
        except Exception:
            features.update({"bls_power": np.nan, "bls_period": np.nan, "bls_duration": np.nan, "bls_depth_ppm": np.nan, "bls_depth_snr": np.nan})
    else:
        features.update({"bls_power": np.nan, "bls_period": np.nan, "bls_duration": np.nan, "bls_depth_ppm": np.nan, "bls_depth_snr": np.nan})

    return features


In [11]:
# ==========================================
# 3. Dataset Builder & Evaluation Utils
# ==========================================

def build_event_feature_table(labels_df, max_systems=None, max_points=6_000):
    """Builds the complete feature matrix and target labels for all classes."""
    event_labels = {0: "neither", 1: "flare", 2: "transit", 3: "both"}
    
    source_df = labels_df[labels_df["anomaly_class"].isin(event_labels.keys())].copy()
    if max_systems is not None:
        source_df = source_df.head(max_systems)

    rows = []
    for _, row in tqdm(source_df.iterrows(), total=len(source_df), desc="Extracting features"):
        system_id = row["system_id"]
        time, flux = load_classifier_lightcurve(f"outputs/{system_id}/")
        
        if time is None:
            continue

        try:
            features = extract_all_features(time, flux, max_points=max_points)
            features.update({
                "system_id": system_id,
                "event_class": event_labels[int(row["anomaly_class"])],
                "anomaly_class": int(row["anomaly_class"]),
            })
            rows.append(features)
        except Exception as exc:
            print(f"Skipping {system_id}: {exc}")

    return pd.DataFrame(rows)

def add_confidence_columns(prediction_df, class_names):
    """Add max probability, probability margin, and normalized entropy confidence diagnostics."""
    prob_cols = [f"p_{name}" for name in class_names]
    probs = prediction_df[prob_cols].to_numpy(dtype=float)
    sorted_probs = np.sort(probs, axis=1)
    
    top = sorted_probs[:, -1]
    second = sorted_probs[:, -2] if probs.shape[1] > 1 else np.zeros_like(top)
    entropy = -np.sum(np.clip(probs, 1e-12, 1.0) * np.log(np.clip(probs, 1e-12, 1.0)), axis=1)
    max_entropy = np.log(probs.shape[1])

    prediction_df["confidence"] = top
    prediction_df["confidence_margin"] = top - second
    prediction_df["confidence_entropy"] = 1.0 - entropy / max_entropy
    return prediction_df


In [14]:
# ==========================================
# 4. Training Pipeline Execution
# ==========================================

# Assuming 'df' is your combined labels dataframe loaded previously
event_feature_df = build_event_feature_table(df, max_systems=None, max_points=50_000)

print(f"\nExtracted Features Shape: {event_feature_df.shape}")
print(event_feature_df["event_class"].value_counts())

event_feature_cols = [col for col in event_feature_df.columns if col not in {"system_id", "event_class", "anomaly_class"}]
X_event = event_feature_df[event_feature_cols]
y_event = event_feature_df["event_class"]

X_event_train, X_event_test, y_event_train, y_event_test, event_ids_train, event_ids_test = train_test_split(
    X_event, y_event, event_feature_df["system_id"], test_size=0.25, random_state=42, stratify=y_event
)

# Initialize and train the Random Forest
event_clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )),
])

Extracting features:   0%|          | 8/2000 [00:41<2:53:25,  5.22s/it]


KeyboardInterrupt: 

In [ ]:
event_clf.fit(X_event_train, y_event_train)
event_classes = list(event_clf.named_steps["model"].classes_)

# Predictions and probabilities
event_proba = event_clf.predict_proba(X_event_test)
event_pred = event_clf.predict(X_event_test)

print("\nConfusion Matrix (Rows=True, Columns=Predicted):")
print(pd.DataFrame(
    confusion_matrix(y_event_test, event_pred, labels=event_classes),
    index=[f"true_{name}" for name in event_classes],
    columns=[f"pred_{name}" for name in event_classes],
))

print("\nClassification Report:")
print(classification_report(y_event_test, event_pred, labels=event_classes))

# Compile results dataframe
probability_df = pd.DataFrame(event_proba, columns=[f"p_{name}" for name in event_classes])

event_results_df = pd.concat([
    pd.DataFrame({
        "system_id": event_ids_test.values,
        "true_event_class": y_event_test.values,
        "pred_event_class": event_pred,
    }).reset_index(drop=True),
    probability_df.reset_index(drop=True),
], axis=1)

event_results_df = add_confidence_columns(event_results_df, event_classes)
event_results_df = event_results_df.sort_values("confidence", ascending=False)

# Optional: Master diagnostic table to merge physical params back in
full_analysis = pd.merge(event_results_df, df, on="system_id", how="inner")